# SABER Feature EDA — GS & COST

Data source: Financial Modeling Prep (FMP) stable API  
Scope: Balance sheet, income statement, cash flow, daily prices, key metrics, macro indicators

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

API_KEY = 'AQ7vZFufNFKFggLRz4E8wpBmEETEBcy9'
BASE    = 'https://financialmodelingprep.com/stable'
TICKERS = ['GS', 'COST']

In [ ]:
# ── helpers ──────────────────────────────────────────────
def fmp_get(endpoint, **params):
    """Fetch JSON from FMP stable API."""
    params['apikey'] = API_KEY
    r = requests.get(f'{BASE}/{endpoint}', params=params)
    r.raise_for_status()
    data = r.json()
    return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame([data])

def camel_to_snake(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

def snakeify(df):
    df.columns = [camel_to_snake(c) for c in df.columns]
    return df

## 1. Data Fetching

In [ ]:
# ── Company profiles ─────────────────────────────────────
profiles = pd.concat(
    [fmp_get('profile', symbol=t) for t in TICKERS], ignore_index=True
)
profiles = snakeify(profiles)
print(f'Profiles: {profiles.shape}')

# show available columns so we pick the right names
profile_show = [c for c in ['symbol', 'company_name', 'sector', 'industry',
    'country', 'market_cap', 'mkt_cap', 'exchange'] if c in profiles.columns]
profiles[profile_show].head()

In [ ]:
# ── Quarterly financial statements (full history) ─────────
bs_list, inc_list, cf_list = [], [], []

for t in TICKERS:
    bs_list.append(fmp_get('balance-sheet-statement', symbol=t, period='quarter', limit=200))
    inc_list.append(fmp_get('income-statement', symbol=t, period='quarter', limit=200))
    cf_list.append(fmp_get('cash-flow-statement', symbol=t, period='quarter', limit=200))

bs_raw  = snakeify(pd.concat(bs_list, ignore_index=True))
inc_raw = snakeify(pd.concat(inc_list, ignore_index=True))
cf_raw  = snakeify(pd.concat(cf_list, ignore_index=True))

print(f'Balance sheet:    {bs_raw.shape}')
print(f'Income statement: {inc_raw.shape}')
print(f'Cash flow:        {cf_raw.shape}')

In [ ]:
# ── Key metrics & ratios (quarterly, full history) ────────
km_list, rat_list = [], []
for t in TICKERS:
    km_list.append(fmp_get('key-metrics', symbol=t, period='quarter', limit=200))
    rat_list.append(fmp_get('ratios', symbol=t, period='quarter', limit=200))

km_raw  = snakeify(pd.concat(km_list, ignore_index=True))
rat_raw = snakeify(pd.concat(rat_list, ignore_index=True))

print(f'Key metrics: {km_raw.shape}')
print(f'Ratios:      {rat_raw.shape}')

In [ ]:
# ── Daily prices ──────────────────────────────────────────
px_list = []
for t in TICKERS:
    df = fmp_get('historical-price-eod/full', symbol=t)
    df['ticker'] = t
    px_list.append(df)

prices_raw = snakeify(pd.concat(px_list, ignore_index=True))
prices_raw['date'] = pd.to_datetime(prices_raw['date'])
prices_raw = prices_raw.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f'Daily prices: {prices_raw.shape}')
print(f'Date range: {prices_raw["date"].min().date()} → {prices_raw["date"].max().date()}')
prices_raw.head()

In [ ]:
# ── Macro indicators ──────────────────────────────────────
macro_names = {
    'realGDP':             'real_gdp',
    'realGDPPerCapita':    'real_gdp_per_capita',
    'retailSales':         'retail_sales',
    'durableGoods':        'durables',
    'federalFunds':        'federal_funds_rate',
    'CPI':                 'cpi',
    'inflationRate':       'inflation_expectation',
    'unemploymentRate':    'unemployment',
    'nonFarmPayroll':      'nonfarm_payroll',
}

MACRO_FROM = '2000-01-01'
MACRO_TO   = '2026-12-31'

macro_series = {}
for fmp_name, col_name in macro_names.items():
    try:
        tmp = fmp_get('economic-indicators', name=fmp_name,
                      **{'from': MACRO_FROM, 'to': MACRO_TO})
        tmp['date'] = pd.to_datetime(tmp['date'])
        tmp = tmp.set_index('date')['value'].sort_index().dropna()
        tmp.name = col_name
        macro_series[col_name] = tmp.resample('QE').last()
        print(f'  {col_name:30s} {len(tmp):>5d} raw obs → {len(macro_series[col_name]):>4d} quarters')
    except Exception as e:
        print(f'  {col_name:30s} FAILED: {e}')

# Treasury yield (separate endpoint)
try:
    tsy = fmp_get('treasury-rates', **{'from': MACRO_FROM, 'to': MACRO_TO})
    tsy = snakeify(tsy)
    tsy['date'] = pd.to_datetime(tsy['date'])
    tsy = tsy.set_index('date').sort_index()
    ycol = 'year10' if 'year10' in tsy.columns else tsy.columns[1]
    tsy_q = tsy[ycol].dropna().resample('QE').last()
    tsy_q.name = 'treasury_yield'
    macro_series['treasury_yield'] = tsy_q
    print(f'  {"treasury_yield":30s} {len(tsy):>5d} raw obs → {len(tsy_q):>4d} quarters')
except Exception as e:
    print(f'  treasury_yield FAILED: {e}')

# Combine — all already aligned to quarter-end dates
macro_df = pd.DataFrame(macro_series)
macro_df.index.name = 'date'
macro_df = macro_df.reset_index().sort_values('date').reset_index(drop=True)

print(f'\nMacro combined: {macro_df.shape}')
print(f'NA%: {100*macro_df.isna().mean().mean():.1f}%')
macro_df.tail()

## 2. Tidy Tables — Quick Look

In [ ]:
# ── Balance sheet snapshot ────────────────────────────────
bs_key_cols = ['symbol', 'date', 'period', 'total_assets', 'total_liabilities',
               'total_stockholders_equity', 'total_current_assets',
               'total_current_liabilities', 'cash_and_cash_equivalents',
               'net_debt', 'long_term_debt', 'goodwill', 'inventory',
               'net_receivables', 'retained_earnings']
bs_show = [c for c in bs_key_cols if c in bs_raw.columns]
bs_raw[bs_show].head(10)

In [ ]:
# ── Income statement snapshot ─────────────────────────────
inc_key_cols = ['symbol', 'date', 'period', 'revenue', 'cost_of_revenue',
                'gross_profit', 'operating_income', 'net_income',
                'ebitda', 'eps', 'eps_diluted', 'interest_expense',
                'income_tax_expense', 'research_and_development_expenses']
inc_show = [c for c in inc_key_cols if c in inc_raw.columns]
inc_raw[inc_show].head(10)

In [ ]:
# ── Cash flow snapshot ────────────────────────────────────
cf_key_cols = ['symbol', 'date', 'period', 'net_income', 'operating_cash_flow',
               'capital_expenditure', 'free_cash_flow',
               'dividends_paid', 'stock_based_compensation',
               'depreciation_and_amortization', 'change_in_working_capital']
cf_show = [c for c in cf_key_cols if c in cf_raw.columns]
cf_raw[cf_show].head(10)

In [ ]:
# ── All available columns per statement ───────────────────
for name, df in [('Balance Sheet', bs_raw), ('Income Statement', inc_raw),
                 ('Cash Flow', cf_raw), ('Key Metrics', km_raw), ('Ratios', rat_raw)]:
    print(f'\n{name} ({df.shape[1]} columns):')
    print(', '.join(sorted(df.columns)))

## 3. Missing Values & Duplicates

In [ ]:
def na_report(df, name):
    """Print NA summary: total, % by column, worst offenders."""
    total_cells = df.shape[0] * df.shape[1]
    total_na = df.isna().sum().sum()
    # also count zeros in numeric columns as potential "soft NAs"
    num_cols = df.select_dtypes(include='number').columns
    total_zeros = (df[num_cols] == 0).sum().sum() if len(num_cols) else 0
    
    print(f'\n{"=" * 60}')
    print(f'{name}: {df.shape[0]} rows × {df.shape[1]} cols')
    print(f'  Total NAs:   {total_na:,} / {total_cells:,}  ({100*total_na/total_cells:.1f}%)')
    print(f'  Total zeros: {total_zeros:,} (in numeric cols only)')
    
    na_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    bad = na_pct[na_pct > 0]
    if len(bad):
        print(f'  Columns with NAs ({len(bad)}):')
        for col, pct in bad.head(15).items():
            print(f'    {col:45s} {pct:5.1f}%')
        if len(bad) > 15:
            print(f'    ... and {len(bad)-15} more')
    else:
        print('  No NAs found!')
    return na_pct

na_bs  = na_report(bs_raw, 'Balance Sheet')
na_inc = na_report(inc_raw, 'Income Statement')
na_cf  = na_report(cf_raw, 'Cash Flow')
na_km  = na_report(km_raw, 'Key Metrics')
na_rat = na_report(rat_raw, 'Ratios')
na_px  = na_report(prices_raw, 'Daily Prices')
na_mac = na_report(macro_df, 'Macro')

In [ ]:
# ── NA heatmap per statement ──────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for ax, (name, df) in zip(axes.flat,
    [('Balance Sheet', bs_raw), ('Income Statement', inc_raw),
     ('Cash Flow', cf_raw), ('Key Metrics', km_raw)]):
    # show only numeric columns with any NAs
    num = df.select_dtypes(include='number')
    has_na = num.columns[num.isna().any()]
    if len(has_na) > 30:
        has_na = has_na[:30]  # cap for readability
    if len(has_na):
        sns.heatmap(num[has_na].isna().T, cbar=False, ax=ax, cmap='Reds')
    ax.set_title(f'{name} — NA pattern')
    ax.set_xlabel('Row')

plt.tight_layout()
plt.show()

In [ ]:
# ── Duplicate check ───────────────────────────────────────
def dup_check(df, name, subset=None):
    n_dup = df.duplicated(subset=subset).sum()
    label = f'(on {subset})' if subset else '(all cols)'
    print(f'{name:25s} {label:30s}: {n_dup} duplicates')
    return n_dup

for name, df, keys in [
    ('Balance Sheet', bs_raw, ['symbol', 'date']),
    ('Income Statement', inc_raw, ['symbol', 'date']),
    ('Cash Flow', cf_raw, ['symbol', 'date']),
    ('Key Metrics', km_raw, ['symbol', 'date']),
    ('Ratios', rat_raw, ['symbol', 'date']),
    ('Daily Prices', prices_raw, ['ticker', 'date']),
]:
    dup_check(df, name, subset=keys)

## 4. Descriptive Statistics

In [ ]:
# ── Balance sheet summary per ticker ──────────────────────
bs_num = bs_raw.select_dtypes(include='number')
key_bs_cols = [c for c in ['total_assets', 'total_liabilities',
    'total_stockholders_equity', 'cash_and_cash_equivalents',
    'net_debt', 'long_term_debt', 'total_current_assets',
    'total_current_liabilities', 'retained_earnings'] if c in bs_num.columns]

for t in TICKERS:
    mask = bs_raw['symbol'] == t
    print(f'\n── {t} Balance Sheet ──')
    display(bs_raw.loc[mask, key_bs_cols].describe().round(0))

In [ ]:
# ── Income statement summary per ticker ───────────────────
key_inc_cols = [c for c in ['revenue', 'gross_profit', 'operating_income',
    'net_income', 'ebitda', 'eps_diluted', 'interest_expense'] if c in inc_raw.columns]

for t in TICKERS:
    mask = inc_raw['symbol'] == t
    print(f'\n── {t} Income Statement ──')
    display(inc_raw.loc[mask, key_inc_cols].describe().round(0))

In [ ]:
# ── Prices summary ────────────────────────────────────────
for t in TICKERS:
    mask = prices_raw['ticker'] == t
    px = prices_raw.loc[mask]
    print(f'\n── {t} Daily Prices ──')
    print(f'  Range: {px["date"].min().date()} → {px["date"].max().date()}  ({len(px)} days)')
    display(px[['open', 'high', 'low', 'close', 'volume']].describe().round(2))

## 5. Basic Plots

In [ ]:
# ── Price time series ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, t in zip(axes, TICKERS):
    px = prices_raw[prices_raw['ticker'] == t]
    ax.plot(px['date'], px['close'], linewidth=0.8)
    ax.set_title(f'{t} — Close Price')
    ax.set_ylabel('USD')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Daily returns distribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, t in zip(axes, TICKERS):
    px = prices_raw[prices_raw['ticker'] == t].copy()
    px['ret'] = px['close'].pct_change()
    ax.hist(px['ret'].dropna(), bins=80, edgecolor='none', alpha=0.7)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(f'{t} — Daily Return Distribution')
    ax.set_xlabel('Return')
    mean_r = px['ret'].mean()
    std_r  = px['ret'].std()
    ax.annotate(f'mean={mean_r:.4f}\nstd={std_r:.4f}', xy=(0.02, 0.95),
                xycoords='axes fraction', va='top', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Volume time series ────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, t in zip(axes, TICKERS):
    px = prices_raw[prices_raw['ticker'] == t]
    ax.bar(px['date'], px['volume'], width=1, alpha=0.5)
    ax.set_title(f'{t} — Daily Volume')
    ax.set_ylabel('Shares')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Key fundamentals over time ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

metrics = [
    ('revenue', 'Revenue'),
    ('net_income', 'Net Income'),
    ('total_assets', 'Total Assets'),
    ('total_liabilities', 'Total Liabilities'),
    ('free_cash_flow', 'Free Cash Flow'),
    ('eps_diluted', 'EPS (diluted)'),
]

for ax, (col, label) in zip(axes.flat, metrics):
    for t in TICKERS:
        # pick the right statement
        for src_name, src in [('inc', inc_raw), ('bs', bs_raw), ('cf', cf_raw)]:
            if col in src.columns:
                df = src[src['symbol'] == t].copy()
                df['date'] = pd.to_datetime(df['date'])
                df = df.sort_values('date')
                ax.plot(df['date'], df[col], marker='.', markersize=3, label=t)
                break
    ax.set_title(label)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Key metrics over time ─────────────────────────────────
km_metrics = [
    ('market_cap',              'Market Cap'),
    ('pe_ratio',                'P/E Ratio'),
    ('pb_ratio',                'P/B Ratio'),
    ('ev_to_ebitda',            'EV/EBITDA'),
    ('roe',                     'ROE'),
    ('roic',                    'ROIC'),
    ('dividend_yield',          'Dividend Yield'),
    ('debt_to_equity',          'Debt/Equity'),
    ('free_cash_flow_per_share','FCF per Share'),
    ('revenue_per_share',       'Revenue per Share'),
    ('net_income_per_share',    'Net Income per Share'),
    ('book_value_per_share',    'Book Value per Share'),
]

# filter to columns that actually exist
km_available = [(col, label) for col, label in km_metrics if col in km_raw.columns]
n = len(km_available)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
axes_flat = axes.flat

for i, (col, label) in enumerate(km_available):
    for t in TICKERS:
        sub = km_raw[km_raw['symbol'] == t].copy()
        sub['date'] = pd.to_datetime(sub['date'])
        sub = sub.sort_values('date')
        axes_flat[i].plot(sub['date'], sub[col], marker='.', markersize=2, label=t)
    axes_flat[i].set_title(label)
    axes_flat[i].legend(fontsize=8)
    axes_flat[i].grid(alpha=0.3)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Macro indicators over time ────────────────────────────
macro_plot_cols = [c for c in macro_df.columns if c != 'date']
n = len(macro_plot_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
axes = axes.flat

for i, col in enumerate(macro_plot_cols):
    tmp = macro_df[['date', col]].dropna()
    axes[i].plot(tmp['date'], tmp[col], linewidth=0.8)
    axes[i].set_title(col)
    axes[i].grid(alpha=0.3)

for j in range(i + 1, len(axes.base.flat)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 6. Column Mapping — FMP vs SABER Skeleton

In [ ]:
# ── Show which SABER skeleton features exist in FMP data ──
from saber_feature_skeleton import (
    BALANCE_SHEET_FEATURES, CASHFLOW_FEATURES,
    INCOME_STATEMENT_FEATURES, FUNDAMENTAL_RATIO_FEATURES,
    MACRO_FEATURES,
)

def coverage_report(saber_cols, fmp_cols, name):
    fmp_set = set(fmp_cols)
    matched = [c for c in saber_cols if c in fmp_set]
    missing = [c for c in saber_cols if c not in fmp_set]
    pct = 100 * len(matched) / len(saber_cols) if saber_cols else 0
    print(f'\n{name}: {len(matched)}/{len(saber_cols)} matched ({pct:.0f}%)')
    if missing:
        print(f'  Missing: {missing[:15]}')
        if len(missing) > 15:
            print(f'  ... +{len(missing)-15} more')
    # also show FMP columns NOT in skeleton
    extra = sorted(set(fmp_cols) - set(saber_cols) - {'symbol', 'date', 'period',
        'reported_currency', 'cik', 'filling_date', 'accepted_date',
        'calendar_year', 'link', 'final_link'})
    if extra:
        print(f'  Extra FMP cols not in skeleton: {extra[:10]}')

coverage_report(BALANCE_SHEET_FEATURES, bs_raw.columns, 'Balance Sheet')
coverage_report(CASHFLOW_FEATURES, cf_raw.columns, 'Cash Flow')
coverage_report(INCOME_STATEMENT_FEATURES, inc_raw.columns, 'Income Statement')
coverage_report(MACRO_FEATURES, macro_df.columns if macro_df is not None else [], 'Macro')

In [ ]:
# ── Correlation matrix for key ratios ─────────────────────
ratio_cols = [c for c in ['pe_ratio_t_t_m', 'pb_ratio', 'debt_equity_ratio',
    'current_ratio', 'return_on_equity', 'return_on_assets',
    'net_profit_margin', 'gross_profit_margin', 'dividend_yield',
    'price_to_free_cash_flows_ratio'] if c in rat_raw.columns]

if ratio_cols:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, t in zip(axes, TICKERS):
        subset = rat_raw[rat_raw['symbol'] == t][ratio_cols].dropna(axis=1, how='all')
        if not subset.empty:
            corr = subset.corr()
            sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
                        center=0, ax=ax, vmin=-1, vmax=1)
        ax.set_title(f'{t} — Ratio Correlations')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Summary: data inventory ───────────────────────────────
inventory = pd.DataFrame([
    {'Dataset': 'Balance Sheet', 'Rows': len(bs_raw), 'Cols': bs_raw.shape[1],
     'NA%': f"{100*bs_raw.isna().mean().mean():.1f}%"},
    {'Dataset': 'Income Statement', 'Rows': len(inc_raw), 'Cols': inc_raw.shape[1],
     'NA%': f"{100*inc_raw.isna().mean().mean():.1f}%"},
    {'Dataset': 'Cash Flow', 'Rows': len(cf_raw), 'Cols': cf_raw.shape[1],
     'NA%': f"{100*cf_raw.isna().mean().mean():.1f}%"},
    {'Dataset': 'Key Metrics', 'Rows': len(km_raw), 'Cols': km_raw.shape[1],
     'NA%': f"{100*km_raw.isna().mean().mean():.1f}%"},
    {'Dataset': 'Ratios', 'Rows': len(rat_raw), 'Cols': rat_raw.shape[1],
     'NA%': f"{100*rat_raw.isna().mean().mean():.1f}%"},
    {'Dataset': 'Daily Prices', 'Rows': len(prices_raw), 'Cols': prices_raw.shape[1],
     'NA%': f"{100*prices_raw.isna().mean().mean():.1f}%"},
    {'Dataset': 'Macro', 'Rows': len(macro_df), 'Cols': macro_df.shape[1],
     'NA%': f"{100*macro_df.isna().mean().mean():.1f}%"},
])
display(inventory)